# Transformer for Translation: English to Spanish

Building a **complete Transformer** (Encoder + Decoder) from scratch to translate English sentences to Spanish. 

This demonstrates the full architecture from the paper *"Attention is All You Need"*.

**We'll learn:**
- How the **Encoder** processes the source language (English)
- How the **Decoder** generates the target language (Spanish)
- The role of **Cross-Attention** in connecting encoder and decoder
- **Positional Encoding** for sequence order
- **Teacher Forcing** (training) vs. **Auto-regressive Decoding** (inference)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import numpy as np

## Transformers are trained using both input and output sequences to learn translation, but once trained, they can generate translations using only the input sentence.

## 1. Dataset: English-Spanish Sentence Pairs

We create a small synthetic dataset for translation purpose. **Verify the dataset is correct?.**

In [ ]:
# English-Spanish parallel corpus
data = [
    ("I love AI", "Me encanta la IA"),
    ("I love machine learning", "Me encanta el aprendizaje automático"),
    ("The cat is black", "El gato es negro"),
    ("The dog is white", "El perro es blanco"),
    ("She studies mathematics", "Ella estudia matemáticas"),
    ("He studies physics", "Él estudia física"),
    ("We learn English", "Nosotros aprendemos inglés"),
    ("They learn Spanish", "Ellos aprenden español"),
    ("The book is good", "El libro es bueno"),
    ("The movie is bad", "La película es mala"),
    ("I eat an apple", "Yo como una manzana"),
    ("She drinks water", "Ella bebe agua"),
    ("He reads a book", "Él lee un libro"),
    ("We write code", "Nosotros escribimos código"),
    ("They play soccer", "Ellos juegan fútbol"),
    ("The house is big", "La casa es grande"),
    ("The car is small", "El coche es pequeño"),
    ("I like music", "Me gusta la música"),
    ("She likes art", "A ella le gusta el arte"),
    ("We have a dog", "Nosotros tenemos un perro")
]

print(f"Dataset size: {len(data)} sentence pairs")
print("\nExample:")
print(f"English: {data[0][0]}")
print(f"Spanish: {data[0][1]}")

## 2. Building Vocabularies

We need **two separate vocabularies**: one for English (source) and one for Spanish (target).

**Special Tokens:**
- `<PAD>`: Padding token (for batching)
- `<SOS>`: Start of sequence (decoder input)
- `<EOS>`: End of sequence (marks completion)

In [ ]:
def build_vocab(sentences):
    vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}
    for sent in sentences:
        for word in sent.lower().split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

# Build vocabularies
en_sents = [pair[0] for pair in data]
es_sents = [pair[1] for pair in data]

en_vocab = build_vocab(en_sents)
es_vocab = build_vocab(es_sents)

# Inverse vocabularies for decoding
eng_inv = {i: w for w, i in en_vocab.items()}
esp_inv = {i: w for w, i in es_vocab.items()}

print(f"English vocabulary size: {len(en_vocab)}")
print(f"Spanish vocabulary size: {len(es_vocab)}")
print(f"\nEnglish vocab sample: {list(en_vocab.items())}")
print(f"Spanish vocab sample: {list(es_vocab.items())}")

## 3. Tokenization: Converting Sentences to Indices

In [ ]:
def encode(sentence, vocab):
    return [vocab[w.lower()] for w in sentence.split()]

def decode(indices, inv_vocab):
    return " ".join([inv_vocab[i] for i in indices if i > 2])  # Skip special tokens

# Example
eng_example = "I love AI"
eng_encoded = encode(eng_example, en_vocab)
print(f"English: {eng_example}")
print(f"Encoded: {eng_encoded}")
print(f"Decoded: {decode(eng_encoded, eng_inv)}")

## 4. Positional Encoding

Transformers process all words in parallel, so they need explicit position information.

**Formula:**
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

This creates unique "signatures" for each position.

In [ ]:
class PositionalEncoding(nn.Module):
    """Positional Encoding"""
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

pe = PositionalEncoding(d_m=64, max_len=50)

## 5. Multi-Head Attention

The core mechanism that allows words to "attend" to each other.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_m, n_heads):
        super().__init__()
        assert d_m % n_heads == 0
        self.d_k = d_m // n_heads
        self.n_h = n_heads
        
        self.W_q = nn.Linear(d_m, d_m)
        self.W_k = nn.Linear(d_m, d_m)
        self.W_v = nn.Linear(d_m, d_m)
        self.W_o = nn.Linear(d_m, d_m)
    
    def forward(self, q, k, v, mask=None):
        bs = q.size(0)
        
        # Linear projections and split into heads
        q = self.W_q(q).view(bs, -1, self.n_h, self.d_k).transpose(1, 2)
        k = self.W_k(k).view(bs, -1, self.n_h, self.d_k).transpose(1, 2)
        v = self.W_v(v).view(bs, -1, self.n_h, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        
        # Concatenate heads
        out = out.transpose(1, 2).contiguous().view(bs, -1, self.n_h * self.d_k)
        return self.W_o(out)

## 6. Feed-Forward Network

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_m, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_m, d_ff)
        self.fc2 = nn.Linear(d_ff, d_m)
    
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

## 7. Encoder Layer

**Process:**
1. Multi-Head Self-Attention (English words attend to each other)
2. Add & Norm
3. Feed-Forward Network
4. Add & Norm

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_m, n_h, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_m, n_h)
        self.ff = FeedForward(d_m, d_ff)
        self.norm1 = nn.LayerNorm(d_m)
        self.norm2 = nn.LayerNorm(d_m)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention
        attn_out = self.attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Feed-forward
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x

## 8. Decoder Layer

**Process:**
1. **Masked** Self-Attention (Spanish words can only see previous Spanish words)
2. Add & Norm
3. **Cross-Attention** (Spanish words attend to English encoder output)
4. Add & Norm
5. Feed-Forward Network
6. Add & Norm

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        # Masked self-attention
        self_attn_out = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_out))
        
        # Cross-attention (attend to encoder output)
        cross_attn_out = self.cross_attn(x, enc_out, enc_out, src_mask)
        x = self.norm2(x + self.dropout(cross_attn_out))
        
        # Feed-forward
        ff_out = self.ff(x)
        x = self.norm3(x + self.dropout(ff_out))
        return x

## 9. Complete Transformer Model

**Architecture:**
```
English → Embedding → Positional Encoding → Encoder Layers → Encoder Output
                                                                    ↓
Spanish → Embedding → Positional Encoding → Decoder Layers ← Cross-Attention
                                                    ↓
                                            Linear → Softmax → Prediction
```

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_sz, tgt_vocab_sz, d_model=128, n_heads=4, no_enc=2, no_dec=2, d_ff=256, dropout=0.1):
        super().__init__()
        
        # Embeddings
        self.src_emb = nn.Embedding(src_vocab_sz, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab_sz, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        
        # Encoder and Decoder stacks
        self.enc_layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(no_enc)])
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(no_dec)])
        
        # Output projection
        self.fc_out = nn.Linear(d_model, tgt_vocab_sz)
        self.dropout = nn.Dropout(dropout)
        
    def make_tgt_mask(self, tgt): # Masked-MHA
        # Causal mask: prevent attending to future tokens 
        sz = tgt.size(1)
        mask = torch.tril(torch.ones(sz, sz)).unsqueeze(0).unsqueeze(0)
        return mask.to(tgt.device)
    
    def encode(self, src):
        x = self.dropout(self.pos_enc(self.src_emb(src)))
        for layer in self.enc_layers:
            x = layer(x)
        return x
    
    def decode(self, tgt, enc_out, tgt_mask):
        x = self.dropout(self.pos_enc(self.tgt_emb(tgt)))
        for layer in self.dec_layers:
            x = layer(x, enc_out, tgt_mask=tgt_mask)
        return self.fc_out(x)
    
    def forward(self, src, tgt):
        tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.encode(src)
        out = self.decode(tgt, enc_out, tgt_mask)
        return out

## 10. Preparing Training Data

**Key Concept: Teacher Forcing**

During training:
- **Decoder Input**: `<SOS> Me encanta la IA`
- **Target Output**: `Me encanta la IA <EOS>`

We feed the correct Spanish words as input, even if the model predicts wrong.

In [ ]:
def prepare_data(data, en_vocab, es_vocab):
    pairs = []
    for en, es in data:
        src = torch.tensor(encode(en, en_vocab), dtype=torch.long)
        tgt_in = torch.tensor([es_vocab["<SOS>"]] + encode(es, es_vocab), dtype=torch.long)
        tgt_out = torch.tensor(encode(es, es_vocab) + [es_vocab["<EOS>"]], dtype=torch.long)
        pairs.append((src, tgt_in, tgt_out))
    return pairs

train_data = prepare_data(data, en_vocab, es_vocab)

# Example
src, tgt_in, tgt_out = train_data[0]
print(f"Source (English): {decode(src.tolist(), en_inv)}")
print(f"Target Input: {[es_inv[i] for i in tgt_in.tolist()]}")
print(f"Target Output: {[es_inv[i] for i in tgt_out.tolist()]}")

## 11. Training the Transformer

**Settings optimized for CPU/Colab:**
- Small model size (`d_model=128`)
- Few layers (2 encoder + 2 decoder)
- Small dataset (20 sentences)
- Should train in ~2-3 minutes on CPU

In [ ]:
# Initialize model
model = Transformer(
    src_vocab_sz=len(en_vocab),
    tgt_vocab_sz=len(es_vocab),
    d_m=128,
    n_heads=4,
    no_enc=2,
    no_dec=2,
    d_ff=256,
    dropout=0.1
)

optz = optim.Adam(model.parameters(), lr=0.0003)
crit = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training loop
losses = []
epochs = 300

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for src, tgt_in, tgt_out in train_data:
        src = src.unsqueeze(0)  # Add batch dimension
        tgt_in = tgt_in.unsqueeze(0)
        tgt_out = tgt_out.unsqueeze(0)
        
        optz.zero_grad()
        out = model(src, tgt_in)
        
        loss = crit(out.view(-1, len(es_vocab)), tgt_out.view(-1))
        loss.backward()
        optz.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_data)
    losses.append(avg_loss)
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

In [ ]:
# Plot training loss

plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.show()

# Save model
torch.save(model.state_dict(), 'transformer_en_es.pth')
print("\nModel saved!")

## 12. Inference: Translating New Sentences

**Auto-regressive Decoding:**
1. Encode the English sentence once
2. Start with `<SOS>` token
3. Decoder predicts next Spanish word
4. Append prediction to input
5. Repeat until `<EOS>` or max length

In [ ]:
def translate(sentence, model, en_vocab, es_vocab, es_inv, max_len=20):
    model.eval()
    
    # Encode source
    src = torch.tensor(encode(sentence, en_vocab), dtype=torch.long).unsqueeze(0)
    enc_out = model.encode(src)
    
    # Start with <SOS>
    tgt = torch.tensor([[es_vocab["<SOS>"]]], dtype=torch.long)
    
    for _ in range(max_len):
        tgt_mask = model.make_tgt_mask(tgt)
        
        with torch.no_grad():
            out = model.decode(tgt, enc_out, tgt_mask)
        
        # Get next token
        next_token = out[:, -1, :].argmax(dim=-1).item()
        
        if next_token == es_vocab["<EOS>"]:
            break
        
        # Append to target
        tgt = torch.cat([tgt, torch.tensor([[next_token]])], dim=1)
    
    # Decode to words
    translation = " ".join([es_inv[i] for i in tgt[0].tolist() if i > 2])
    return translation

## 13. Testing Translations

In [ ]:
# Test on training examples
test_sentences = [
    "I love AI",
    "The cat is black",
    "She studies mathematics",
    "We learn English",
    "I eat an apple"
]

print("TRANSLATIONS")
for sent in test_sentences:
    translation = translate(sent, model, en_vocab, es_vocab, es_inv)
    print(f"English:  {sent}")
    print(f"Spanish:  {translation} \n")


## 14. Understanding the Architecture

### Key Differences:

| Component | Encoder | Decoder |
|-----------|---------|----------|
| **Self-Attention** | Bidirectional (sees all words) | Causal (only sees previous words) |
| **Cross-Attention** | No | Yes (attends to encoder output) |
| **Purpose** | Understand source language | Generate target language |

### Why This Works:

1. **Encoder** creates rich, context-aware representations of English words
2. **Decoder** generates Spanish words one at a time
3. **Cross-Attention** allows decoder to "look at" relevant English words when generating each Spanish word
4. **Positional Encoding** preserves word order information

### Training vs. Inference:

- **Training (Teacher Forcing)**: Feed correct Spanish words as input -> Fast, stable
- **Inference (Auto-regressive)**: Feed model's own predictions -> Slower, but generates new translations

## 15. Experiment: Visualizing Cross-Attention

Let's see which English words the decoder "looks at" when generating Spanish words.

In [ ]:
def get_cross_attention(sentence, model, en_vocab, es_vocab, es_inv):
    model.eval()
    
    # Encode source
    src = torch.tensor(encode(sentence, en_vocab), dtype=torch.long).unsqueeze(0)
    enc_out = model.encode(src)
    
    # Generate translation and collect attention
    tgt = torch.tensor([[es_vocab["<SOS>"]]], dtype=torch.long)
    translations = []
    
    for _ in range(10):
        tgt_mask = model.make_tgt_mask(tgt)
        
        with torch.no_grad():
            out = model.decode(tgt, enc_out, tgt_mask)
        
        next_token = out[:, -1, :].argmax(dim=-1).item()
        if next_token == es_vocab["<EOS>"]:
            break
        
        translations.append(es_inv[next_token])
        tgt = torch.cat([tgt, torch.tensor([[next_token]])], dim=1)
    
    return sentence.split(), translations

# Example
en_words, es_words = get_cross_attention("I love AI", model, en_vocab, es_vocab, es_inv)
print(f"English: {' '.join(en_words)}")
print(f"Spanish: {' '.join(es_words)}")